# Plugin/Webhook-Service — Webhook-Lebenszyklus

Dieses Notebook demonstriert den Plugin/Webhook-Service (Port 8006) — den **Output-Adapter**
von TeamBoard: er übersetzt Domain-Events in ausgehende, **HMAC-signierte** HTTP-Zustellungen
an externe Endpunkte (mit Retry und Zustellungs-Historie).

Durchlaufen wird der volle Lebenszyklus einer Webhook-Registrierung:
registrieren → Signatur verifizieren → Test-Zustellung auslösen → Deliveries inspizieren →
aktualisieren → deaktivieren/aktivieren → Secret rotieren → löschen.

Webhooks sind **projekt-gebunden**: Registrierung/Listing laufen über
`/api/v1/projects/{projectId}/webhooks`, alle Operationen an einem bestehenden Webhook über
`/api/v1/webhooks/{webhookId}`. Das Gateway routet beide Formen an den Plugin-Service.

## Voraussetzungen

1. Stack läuft und Demo-Daten sind vorhanden:
   ```bash
   make up && make seed
   ```
   `make seed` legt `alice@teamboard.local` und ein Projekt 'Demo Project' an — beides nutzt
   dieses Notebook.
2. Python-Paket `requests` (`pip install requests`).
3. Ein Empfänger für die Zustellungen. Am einfachsten eine Wegwerf-URL von
   [webhook.site](https://webhook.site) — kopiere deine eindeutige URL unten in `TARGET_URL`.
   Im lokalen Dev-Setup ist `ALLOW_PRIVATE_URLS=true` gesetzt, d. h. auch ein lokaler Empfänger
   (z. B. `http://host.docker.internal:9000`) ist erlaubt.

## Setup

In [2]:
import hashlib
import hmac
import json
import os
import time

import requests

# Das Gateway (Traefik) routet /api/v1/webhooks und /api/v1/projects/{id}/webhooks an den
# Plugin-Service.
BASE_URL = os.environ.get("API_BASE_URL", "http://localhost")

# Seed-Zugangsdaten (aus: make seed)
EMAIL = "alice@teamboard.local"
PASSWORD = os.environ.get("SEED_ALICE_PASSWORD", "AliceSecret123!")

# Ziel-URL der Webhook-Zustellungen — durch deine eigene ersetzen:
TARGET_URL = "https://webhook.site/YOUR-UNIQUE-ID"

print(f"API base: {BASE_URL}")

API base: http://localhost


## 1. Authentifizieren

In [3]:
resp = requests.post(f"{BASE_URL}/api/v1/auth/login", json={
    "email": EMAIL,
    "password": PASSWORD,
})
resp.raise_for_status()

access_token = resp.json()["data"]["access_token"]
headers = {"Authorization": f"Bearer {access_token}"}
print("Eingeloggt, Token erhalten.")

Eingeloggt, Token erhalten.


## 2. Projekt auswählen

In [4]:
resp = requests.get(f"{BASE_URL}/api/v1/projects", headers=headers)
resp.raise_for_status()

projects = resp.json()["data"]
project_id = projects[0]["id"]
print(f"Verwende Projekt: {projects[0]['name']}  ({project_id})")

Verwende Projekt: CT2  (5df30720-836a-4706-8532-519863a334ac)


## 3. Webhook registrieren

**Event-Filter-Muster**
- `"*"` — jedes Event
- `"task.*"` — jedes Event, dessen Typ mit `task.` beginnt
- `"task.created"` — exakter Typ-Match

Das `secret` wird **nur einmal** (bei der Registrierung) zurückgegeben — jetzt sichern.

In [5]:
resp = requests.post(
    f"{BASE_URL}/api/v1/projects/{project_id}/webhooks",
    headers=headers,
    json={
        "target_url": TARGET_URL,
        "description": "Example webhook — task and member events",
        "event_filter": ["task.*", "project.member.added"],
    },
)
resp.raise_for_status()

data = resp.json()["data"]
webhook_id = data["id"]
secret = data["secret"]  # nur einmal zurückgegeben — jetzt speichern

print(json.dumps(data, indent=2))
print(f"\nwebhook_id : {webhook_id}")
print(f"secret     : {secret}")

{
  "id": "cb33283c-7c05-442d-a89b-27f12bd54faa",
  "project_id": "5df30720-836a-4706-8532-519863a334ac",
  "target_url": "https://webhook.site/YOUR-UNIQUE-ID",
  "description": "Example webhook \u2014 task and member events",
  "event_filter": [
    "task.*",
    "project.member.added"
  ],
  "active": true,
  "created_by": "105a5e9d-1749-41e9-9fb3-71c82ef35ba4",
  "created_at": "2026-07-01T10:57:44.774256Z",
  "updated_at": "2026-07-01T10:57:44.774256Z",
  "secret": "f0b0a2bdbd6e169fd7ef15a01fc7bd8939f6cc2fa36c7616f32677efd6d92be6"
}

webhook_id : cb33283c-7c05-442d-a89b-27f12bd54faa
secret     : f0b0a2bdbd6e169fd7ef15a01fc7bd8939f6cc2fa36c7616f32677efd6d92be6


## 4. Eingehende Zustellung verifizieren

TeamBoard signiert jede ausgehende POST-Zustellung mit:

```
HMAC-SHA256(key=secret, msg="{timestamp}.{body}")
```

Header, die jede Zustellung trägt:
- `X-TeamBoard-Signature: sha256=<hex>`
- `X-TeamBoard-Timestamp: <unix_seconds>`
- `X-TeamBoard-Event: <event_type>` (z. B. `task.created`)
- `X-TeamBoard-Event-Id: <uuid>`
- `X-TeamBoard-Delivery-Id: <uuid>`

Dein Empfänger sollte die Signatur **konstant-zeitlich** vergleichen (s. u.).

In [6]:
def verify_signature(secret: str, body: bytes, signature: str, timestamp: str) -> bool:
    """Gibt True zurück, wenn die Zustellungs-Signatur gültig ist."""
    msg = f"{timestamp}.".encode() + body
    expected = "sha256=" + hmac.new(secret.encode(), msg, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, signature)


# Simuliert, was dein Webhook-Endpunkt empfängt:
body = b'{"event_type":"task.created","data":{"id":"abc"}}'
timestamp = str(int(time.time()))

# Signatur exakt so berechnen, wie TeamBoard es tut:
msg = f"{timestamp}.".encode() + body
sig = "sha256=" + hmac.new(secret.encode(), msg, hashlib.sha256).hexdigest()

print("Signatur gültig:", verify_signature(secret, body, sig, timestamp))

Signatur gültig: True


## 5. Test-Zustellung auslösen

Sendet einen synthetischen Ping an die `target_url` und legt einen Delivery-Datensatz an.

In [7]:
resp = requests.post(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}/test",
    headers=headers,
)
resp.raise_for_status()
print(json.dumps(resp.json(), indent=2))

{
  "data": {
    "delivery_id": "19221817-c244-4e88-92d4-c1d9b4906fa4"
  }
}


## 6. Zustellungen auflisten

In [8]:
resp = requests.get(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}/deliveries",
    headers=headers,
)
resp.raise_for_status()

deliveries = resp.json()["data"]
print(f"{len(deliveries)} Zustellung(en) gefunden")
for d in deliveries:
    print(f"  {d['id']}  status={d['status']}  attempts={d['attempt_count']}  http={d.get('last_response_status')}")

1 Zustellung(en) gefunden
  19221817-c244-4e88-92d4-c1d9b4906fa4  status=dead  attempts=1  http=None


## 7. Einzelne Zustellung inspizieren

In [9]:
if deliveries:
    delivery_id = deliveries[0]["id"]
    resp = requests.get(
        f"{BASE_URL}/api/v1/webhooks/{webhook_id}/deliveries/{delivery_id}",
        headers=headers,
    )
    resp.raise_for_status()
    print(json.dumps(resp.json()["data"], indent=2))

{
  "id": "19221817-c244-4e88-92d4-c1d9b4906fa4",
  "webhook_id": "cb33283c-7c05-442d-a89b-27f12bd54faa",
  "event_id": "test-8674a5a8-1069-471d-a80e-19a4d805268e",
  "event_type": "webhook.test",
  "status": "dead",
  "attempt_count": 1,
  "last_error": "HTTP 404",
  "last_attempted_at": "2026-07-01T10:58:05.963335Z",
  "failed_permanently_at": "2026-07-01T10:58:05.963335Z",
  "created_at": "2026-07-01T10:58:04.897797Z"
}


## 8. Webhook aktualisieren (Filter und Beschreibung ändern)

In [10]:
resp = requests.patch(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}",
    headers=headers,
    json={
        "description": "Updated — all events",
        "event_filter": ["*"],
    },
)
resp.raise_for_status()
print(json.dumps(resp.json()["data"], indent=2))

{
  "id": "cb33283c-7c05-442d-a89b-27f12bd54faa",
  "project_id": "5df30720-836a-4706-8532-519863a334ac",
  "target_url": "https://webhook.site/YOUR-UNIQUE-ID",
  "description": "Updated \u2014 all events",
  "event_filter": [
    "*"
  ],
  "active": true,
  "created_by": "105a5e9d-1749-41e9-9fb3-71c82ef35ba4",
  "created_at": "2026-07-01T10:57:44.774256Z",
  "updated_at": "2026-07-01T10:58:21.497938Z"
}


## 9. Deaktivieren / aktivieren

In [11]:
requests.post(f"{BASE_URL}/api/v1/webhooks/{webhook_id}/disable", headers=headers).raise_for_status()
print("Webhook deaktiviert.")

requests.post(f"{BASE_URL}/api/v1/webhooks/{webhook_id}/enable", headers=headers).raise_for_status()
print("Webhook wieder aktiviert.")

Webhook deaktiviert.
Webhook wieder aktiviert.


## 10. Secret rotieren

Aktualisiere deinen empfangenden Server **vor** dem Rotieren — das alte Secret wird sofort
ungültig.

In [12]:
resp = requests.post(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}/rotate-secret",
    headers=headers,
)
resp.raise_for_status()

new_secret = resp.json()["data"]["secret"]
secret = new_secret  # lokale Referenz aktualisieren
print(f"Neues Secret: {new_secret}")

Neues Secret: defeae94fe8af472b722be18fca49f0f12aa9b32b1f07ba7df8de65a02264f30


## 11. Alle Webhooks des Projekts auflisten

In [13]:
resp = requests.get(
    f"{BASE_URL}/api/v1/projects/{project_id}/webhooks",
    headers=headers,
)
resp.raise_for_status()
print(json.dumps(resp.json()["data"], indent=2))

[
  {
    "id": "cb33283c-7c05-442d-a89b-27f12bd54faa",
    "project_id": "5df30720-836a-4706-8532-519863a334ac",
    "target_url": "https://webhook.site/YOUR-UNIQUE-ID",
    "description": "Updated \u2014 all events",
    "event_filter": [
      "*"
    ],
    "active": true,
    "created_by": "105a5e9d-1749-41e9-9fb3-71c82ef35ba4",
    "created_at": "2026-07-01T10:57:44.774256Z",
    "updated_at": "2026-07-01T10:58:27.845484Z"
  }
]


## 12. Webhook löschen

In [14]:
resp = requests.delete(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}",
    headers=headers,
)
resp.raise_for_status()
print(f"Webhook {webhook_id} gelöscht.")

Webhook cb33283c-7c05-442d-a89b-27f12bd54faa gelöscht.
